# Daten und Codierung – Bitebene

Dieses Notebook begleitet das Thema **"Daten und Codierung – Bitebene"** und deckt die im Lehrplan
beschriebenen Kompetenzen ab:

**Basisfach**
1. Negative Zahlen im Einerkomplement und Zweierkomplement angeben und interpretieren
2. Addition und Subtraktion von Binärzahlen in Zweierkomplementdarstellung schriftlich durchführen
3. Einschränkungen beim Rechnen mit endlicher Stellenzahl (Overflow, Genauigkeit)
4. Merkmale von Codes (Umkehrbarkeit, Präfixfreiheit, feste/variable Bitlänge)
5. Das Huffmanverfahren als verlustfreies Kompressionsverfahren (Codierung & Decodierung von Hand)

**Leistungsfach (zusätzlich)**
6. Festkommadarstellung als Codierung nichtganzer Zahlen
7. Erweiterungsmöglichkeiten von Textcodierungen (Biterweiterung, Escapezeichen, Codepages)
8. Das LZW-Verfahren als Wörterbuchverfahren
9. Hashfunktionen: Konzept, Anforderungen (Nichtumkehrbarkeit, Kollisionsvermeidung), Anwendungen

> 💡 Jeder Abschnitt enthält: **Theorie**, **von-Hand-Beispiele**, **interaktiven Python-Code** und
> **Übungsaufgaben** zum Selbertesten.


---
## 1. Negative Zahlen: Einerkomplement und Zweierkomplement

### Vorbemerkung: Feste vs. beliebige Bitlänge

Auf **Hardware-Ebene** (CPU-Register) und bei den **primitiven Datentypen** der meisten
Programmiersprachen (z. B. `int`/`long` in Java oder C) werden Zahlen mit **fester Bitlänge**
gespeichert (typischerweise 8, 16, 32 oder 64 Bit). Das ist auch der Grund, warum dort
Overflow-Probleme auftreten können (siehe Abschnitt 2).

⚠️ **Python ist hier eine Ausnahme:** Der Datentyp `int` in Python hat eine **beliebige
Genauigkeit** (arbitrary precision / "Bignum"). Python vergrößert den benötigten Speicher
automatisch, wenn eine Zahl zu groß wird – ein klassischer Integer-Overflow wie in C oder Java
tritt bei Python-`int` praktisch nicht auf. Das folgende Beispiel zeigt das:

```python
import sys
a = 2**1000
print(sys.getsizeof(a))       # 160 Bytes
print(sys.getsizeof(2**10000)) # 1360 Bytes -- deutlich mehr!
```

Für dieses Notebook **simulieren** wir daher bewusst Zahlen mit **fester Bitlänge** (z. B. 8 Bit),
um die Konzepte Zweierkomplement, Overflow usw. zu demonstrieren – so, wie sie auf echter
Hardware bzw. in Sprachen mit festen Ganzzahltypen tatsächlich auftreten.

---

Für **vorzeichenbehaftete** ganze Zahlen mit fester Bitlänge gibt es verschiedene Darstellungen:

### 1.1 Vorzeichen-Betrag-Darstellung
Das höchstwertige Bit (MSB) gibt das Vorzeichen an (0 = positiv, 1 = negativ), der Rest den Betrag.

Beispiel (8 Bit): `+5 = 00000101`, `−5 = 10000101`

**Nachteile:**
- Es gibt zwei Darstellungen für die Null (`00000000` = +0 und `10000000` = −0).
- **Addition und Subtraktion benötigen unterschiedliche Algorithmen** bzw. Fallunterscheidungen:
  Vorzeichen und Betrag müssen getrennt betrachtet werden. Bei zwei Zahlen mit **gleichem**
  Vorzeichen werden die Beträge addiert, bei **unterschiedlichem** Vorzeichen muss der
  Hardware/Algorithmus die Beträge **subtrahieren** und zusätzlich ermitteln, welche Zahl
  betragsmäßig größer ist (um das richtige Vorzeichen des Ergebnisses zu bestimmen). Das macht
  die Schaltungslogik deutlich komplexer als beim Zweierkomplement (siehe unten).


In [1]:
def vorzeichen_betrag_addieren(a: int, b: int) -> int:
    """Demonstriert, dass Vorzeichen-Betrag-Addition eine Fallunterscheidung braucht."""
    if (a >= 0) == (b >= 0):
        # gleiches Vorzeichen -> Betraege einfach addieren
        ergebnis = abs(a) + abs(b)
        return ergebnis if a >= 0 else -ergebnis
    else:
        # unterschiedliches Vorzeichen -> Betraege subtrahieren,
        # zusaetzlich muss ermittelt werden, welcher Betrag groesser ist!
        if abs(a) >= abs(b):
            ergebnis = abs(a) - abs(b)
            return ergebnis if a >= 0 else -ergebnis
        else:
            ergebnis = abs(b) - abs(a)
            return ergebnis if b >= 0 else -ergebnis


print("Vorzeichen-Betrag-Addition benötigt je nach Vorzeichenkombination unterschiedliche Schritte:")
print(f"  5 + 3   (gleiches Vorzeichen)                = {vorzeichen_betrag_addieren(5, 3)}")
print(f" -5 + -3  (gleiches Vorzeichen)                = {vorzeichen_betrag_addieren(-5, -3)}")
print(f"  5 + -3  (unterschiedl. Vorzeichen, |a|>=|b|) = {vorzeichen_betrag_addieren(5, -3)}")
print(f" -5 + 3   (unterschiedl. Vorzeichen, |a|<|b|)  = {vorzeichen_betrag_addieren(-5, 3)}")


Vorzeichen-Betrag-Addition benötigt je nach Vorzeichenkombination unterschiedliche Schritte:
  5 + 3   (gleiches Vorzeichen)                = 8
 -5 + -3  (gleiches Vorzeichen)                = -8
  5 + -3  (unterschiedl. Vorzeichen, |a|>=|b|) = 2
 -5 + 3   (unterschiedl. Vorzeichen, |a|<|b|)  = -2


In [2]:
def to_binary(n: int, bits: int = 8) -> str:
    """Gibt die Zweierkomplement-Darstellung von n mit fester Bitanzahl zurück."""
    if n < 0:
        n = (1 << bits) + n  # Zweierkomplement berechnen
    if not (0 <= n < (1 << bits)):
        raise ValueError(f"{n} passt nicht in {bits} Bit (Zweierkomplement)")
    return format(n, f'0{bits}b')


def ones_complement(n: int, bits: int = 8) -> str:
    """Gibt die Einerkomplement-Darstellung von n mit fester Bitanzahl zurück."""
    if n >= 0:
        return format(n, f'0{bits}b')
    positive = format(-n, f'0{bits}b')
    inverted = ''.join('1' if b == '0' else '0' for b in positive)
    return inverted


def from_twos_complement(bitstring: str) -> int:
    """Interpretiert einen Bitstring als Zweierkomplement-Zahl."""
    bits = len(bitstring)
    value = int(bitstring, 2)
    if bitstring[0] == '1':  # negatives Vorzeichen
        value -= (1 << bits)
    return value


### 1.2 Einerkomplement (Ones' Complement)
Negative Zahlen erhält man, indem man **alle Bits** der positiven Zahl invertiert (0↔1).

Beispiel (8 Bit): `+5 = 00000101` → `−5 = 11111010`

Auch hier gibt es zwei Nullen: `00000000` (+0) und `11111111` (−0).

**Addition beim Einerkomplement:** Man braucht – anders als bei der Vorzeichen-Betrag-Darstellung –
**keine** Fallunterscheidung nach Vorzeichen; man addiert stumpf bitweise. Es ist aber ein
**zusätzlicher Korrekturschritt** nötig: der sogenannte **Übertragswickel (End-Around-Carry)**.
Entsteht beim Addieren ein Übertrag **aus** dem höchstwertigen Bit (MSB) heraus, darf dieser nicht
einfach verworfen werden (wie beim Zweierkomplement), sondern muss **wieder auf das
niedrigstwertige Bit addiert** werden.

**Beispiel: 5 + (−3) im 8-Bit-Einerkomplement**

```
   00000101   ( 5)
 + 11111100   (-3, Einerkomplement von 00000011)
 -----------
  100000001   (9-Bit-Zwischenergebnis, Übertrag = 1)

   00000001
 +        1   <- Übertrag wieder addieren (End-Around-Carry)
 -----------
   00000010   ( 2)  ✓ richtig
```

Damit ist das Einerkomplement eine **Zwischenstufe**: kein Sonderfall nach Vorzeichen wie bei
Vorzeichen-Betrag, aber ein zusätzlicher Korrekturschritt, den das Zweierkomplement gar nicht
braucht (dort wird der Übertrag einfach verworfen, siehe Abschnitt 2). Das ist einer der Gründe,
warum sich in der Praxis das **Zweierkomplement** durchgesetzt hat.


In [3]:
def einerkomplement_addieren(a: int, b: int, bits: int = 8) -> int:
    """Addiert zwei Zahlen im Einerkomplement inkl. End-Around-Carry."""
    mask = (1 << bits) - 1

    a_bits = ones_complement(a, bits)
    b_bits = ones_complement(b, bits)

    zwischensumme = int(a_bits, 2) + int(b_bits, 2)
    uebertrag = zwischensumme >> bits                  # Übertrag aus dem MSB?
    ergebnis_u = (zwischensumme & mask) + uebertrag     # End-Around-Carry addieren

    ergebnis_bits = format(ergebnis_u, f'0{bits}b')
    if ergebnis_bits[0] == '1':
        ergebnis_signed = -(int(''.join('1' if c == '0' else '0' for c in ergebnis_bits), 2))
    else:
        ergebnis_signed = int(ergebnis_bits, 2)

    print(f"  {a_bits}  ({a:>4})")
    print(f"+ {b_bits}  ({b:>4})")
    print(f"{'-' * (bits + 2)}")
    print(f"  {format(zwischensumme & mask, f'0{bits}b')}  (Übertrag = {uebertrag})")
    if uebertrag:
        print(f"+ End-Around-Carry: {uebertrag}")
        print(f"= {ergebnis_bits}  ({ergebnis_signed})")
    else:
        print(f"= {ergebnis_bits}  ({ergebnis_signed})  (kein Übertrag, keine Korrektur nötig)")
    return ergebnis_signed


print("Beispiel: 5 + (-3) im Einerkomplement")
einerkomplement_addieren(5, -3)

print("\nZum Vergleich, ohne Übertrag: 5 + 2")
einerkomplement_addieren(5, 2)


Beispiel: 5 + (-3) im Einerkomplement
  00000101  (   5)
+ 11111100  (  -3)
----------
  00000001  (Übertrag = 1)
+ End-Around-Carry: 1
= 00000010  (2)

Zum Vergleich, ohne Übertrag: 5 + 2
  00000101  (   5)
+ 00000010  (   2)
----------
  00000111  (Übertrag = 0)
= 00000111  (7)  (kein Übertrag, keine Korrektur nötig)


7

### 1.3 Zweierkomplement (Two's Complement)
Negative Zahlen erhält man, indem man **alle Bits invertiert und danach 1 addiert**
(= Einerkomplement + 1).

Beispiel (8 Bit): `+5 = 00000101` → invertieren: `11111010` → +1 → `−5 = 11111011`

**Vorteile des Zweierkomplements** (deshalb wird es in der Praxis fast ausschließlich verwendet):
- Es gibt nur **eine** Darstellung der Null.
- Addition und Subtraktion funktionieren mit **derselben Schaltung/demselben Algorithmus** wie bei
  vorzeichenlosen Zahlen – man addiert einfach bitweise mit Übertrag, ganz ohne Fallunterscheidung
  nach Vorzeichen (wie bei Vorzeichen-Betrag) und ganz ohne Korrekturschritt (wie beim
  Einerkomplement). Der Übertrag aus dem MSB wird einfach verworfen. Das macht die Hardware
  einfacher und schneller.

Bei $n$ Bit im Zweierkomplement lässt sich der Wertebereich beschreiben als:

$$-2^{n-1} \; \text{bis} \; 2^{n-1}-1$$

Für $n=8$: $-128$ bis $127$.


In [4]:
import sys

# Demonstration: Python-int hat KEINE feste Bitlaenge (beliebige Genauigkeit)
a = 2**1000
b = 2**10000

print(f"2**1000  belegt {sys.getsizeof(a):>5} Bytes im Speicher")
print(f"2**10000 belegt {sys.getsizeof(b):>5} Bytes im Speicher")
print("\n--> Der Speicherbedarf wächst mit der Größe der Zahl.")
print("    In C/Java würde ein fester 64-Bit-Typ bei so großen Zahlen längst überlaufen!")


2**1000  belegt   160 Bytes im Speicher
2**10000 belegt  1360 Bytes im Speicher

--> Der Speicherbedarf wächst mit der Größe der Zahl.
    In C/Java würde ein fester 64-Bit-Typ bei so großen Zahlen längst überlaufen!


In [5]:
# --- Beispiele: Zweierkomplement-Darstellung ---
for zahl in [5, -5, 0, 127, -128]:
    try:
        print(f"{zahl:>5} -> Zweierkomplement (8 Bit): {to_binary(zahl, 8)}")
    except ValueError as e:
        print(f"{zahl:>5} -> Fehler: {e}")


    5 -> Zweierkomplement (8 Bit): 00000101
   -5 -> Zweierkomplement (8 Bit): 11111011
    0 -> Zweierkomplement (8 Bit): 00000000
  127 -> Zweierkomplement (8 Bit): 01111111
 -128 -> Zweierkomplement (8 Bit): 10000000


### ✏️ Übung 1
1. Bestimme die 8-Bit-Zweierkomplementdarstellung von **-42**, **17** und **-1**.
2. Interpretiere den Bitstring `10110011` einmal als vorzeichenlose Zahl und einmal als
   Zweierkomplement-Zahl. Vergleiche die Ergebnisse.
3. Warum hat das Einerkomplement zwei Darstellungen der Null, das Zweierkomplement aber nur eine?

Nutze die Zelle unten, um deine Antworten zu überprüfen.


In [6]:
# Übung 1 - Lösung überprüfen
print("Aufgabe 1:")
for zahl in [-42, 17, -1]:
    print(f"  {zahl:>4} -> {to_binary(zahl, 8)}")

print("\nAufgabe 2:")
bitstring = "10110011"
vorzeichenlos = int(bitstring, 2)
zweierkomplement = from_twos_complement(bitstring)
print(f"  {bitstring} als vorzeichenlose Zahl: {vorzeichenlos}")
print(f"  {bitstring} als Zweierkomplement:    {zweierkomplement}")


Aufgabe 1:
   -42 -> 11010110
    17 -> 00010001
    -1 -> 11111111

Aufgabe 2:
  10110011 als vorzeichenlose Zahl: 179
  10110011 als Zweierkomplement:    -77


---
## 2. Addition und Subtraktion im Zweierkomplement

Der große Vorteil des Zweierkomplements: Addition erfolgt **genau wie bei vorzeichenlosen Zahlen** –
man addiert einfach bitweise mit Übertrag (Carry). Subtraktion $a - b$ wird als Addition
$a + (-b)$ realisiert, wobei $-b$ das Zweierkomplement von $b$ ist.

### Beispiel: 5 + (-3) im 8-Bit-Zweierkomplement

```
   00000101   ( 5)
 + 11111101   (-3)
 -----------
   00000010   ( 2)   <- Übertrag aus Bit 7 wird verworfen
```

### 2.1 Overflow (Überlauf)

Ein Overflow tritt auf, wenn das Ergebnis einer Addition/Subtraktion **außerhalb** des
darstellbaren Wertebereichs liegt (z. B. bei 8 Bit außerhalb von −128…127).

**Erkennungsregel:** Ein Overflow liegt vor, wenn der Übertrag **in** das Vorzeichenbit und
der Übertrag **aus** dem Vorzeichenbit unterschiedlich sind. Vereinfacht gesagt:
- Addition zweier **positiver** Zahlen ergibt ein **negatives** Ergebnis → Overflow
- Addition zweier **negativer** Zahlen ergibt ein **positives** Ergebnis → Overflow

### 2.2 Genauigkeit bei endlicher Stellenzahl

Neben dem Overflow bei ganzen Zahlen gibt es bei Gleitkommazahlen zusätzlich das Problem der
**begrenzten Genauigkeit** (Rundungsfehler), da nicht jede reelle Zahl exakt mit endlich vielen
Bits dargestellt werden kann (siehe Abschnitt 6, Festkommadarstellung).


In [7]:
def add_twos_complement(a: int, b: int, bits: int = 8):
    """Addiert zwei Zahlen im Zweierkomplement und erkennt Overflow."""
    mask = (1 << bits) - 1
    a_bits = to_binary(a, bits)
    b_bits = to_binary(b, bits)

    a_u = int(a_bits, 2)
    b_u = int(b_bits, 2)
    result_u = (a_u + b_u) & mask
    result_signed = from_twos_complement(format(result_u, f'0{bits}b'))

    # Overflow: beide Operanden positiv, Ergebnis negativ -- oder umgekehrt
    overflow = False
    if a >= 0 and b >= 0 and result_signed < 0:
        overflow = True
    if a < 0 and b < 0 and result_signed >= 0:
        overflow = True

    print(f"  {a_bits}  ({a:>4})")
    print(f"+ {b_bits}  ({b:>4})")
    print(f"{'-' * (bits + 2)}")
    print(f"  {format(result_u, f'0{bits}b')}  ({result_signed:>4})", end="")
    print("   <-- OVERFLOW!" if overflow else "")
    return result_signed, overflow


print("Beispiel 1: 5 + (-3)")
add_twos_complement(5, -3)

print("\nBeispiel 2: Overflow bei 8 Bit (100 + 50)")
add_twos_complement(100, 50)

print("\nBeispiel 3: Overflow bei negativen Zahlen (-100 + -50)")
add_twos_complement(-100, -50)


Beispiel 1: 5 + (-3)
  00000101  (   5)
+ 11111101  (  -3)
----------
  00000010  (   2)

Beispiel 2: Overflow bei 8 Bit (100 + 50)
  01100100  ( 100)
+ 00110010  (  50)
----------
  10010110  (-106)   <-- OVERFLOW!

Beispiel 3: Overflow bei negativen Zahlen (-100 + -50)
  10011100  (-100)
+ 11001110  ( -50)
----------
  01101010  ( 106)   <-- OVERFLOW!


(106, True)

### ✏️ Übung 2
1. Führe die Subtraktion **12 − 20** im 8-Bit-Zweierkomplement schriftlich (per Hand) durch
   und überprüfe dein Ergebnis mit der Funktion `add_twos_complement(12, -20)`.
2. Finde zwei Zahlen im Bereich −128…127, deren Addition einen Overflow erzeugt, und erkläre,
   warum das Ergebnis falsch ist.
3. Kann bei der Addition einer positiven und einer negativen Zahl ein Overflow auftreten?
   Begründe.


In [8]:
# Übung 2 - Platz zum Ausprobieren
add_twos_complement(12, -20)
add_twos_complement(90, 90)


  00001100  (  12)
+ 11101100  ( -20)
----------
  11111000  (  -8)
  01011010  (  90)
+ 01011010  (  90)
----------
  10110100  ( -76)   <-- OVERFLOW!


(-76, True)

---
## 3. Merkmale von Codes

Bevor wir zur Huffman-Codierung kommen, betrachten wir grundlegende **Eigenschaften von Codes**:

| Merkmal | Bedeutung |
|---|---|
| **Umkehrbarkeit (Eindeutigkeit)** | Aus dem codierten Wort lässt sich die ursprüngliche Nachricht eindeutig rekonstruieren. |
| **Präfixfreiheit** | Kein Codewort ist Anfang (Präfix) eines anderen Codewortes. Das ermöglicht eindeutige Decodierung ohne Trennzeichen. |
| **Feste Bitlänge** | Alle Codewörter haben dieselbe Länge (z. B. ASCII: immer 7/8 Bit). Einfach zu decodieren, aber ineffizient bei ungleicher Häufigkeit der Zeichen. |
| **Variable Bitlänge** | Codewörter können unterschiedlich lang sein (z. B. Morsecode, Huffman-Code). Kann sehr effizient sein, benötigt aber Präfixfreiheit zur eindeutigen Decodierung. |

**Beispiel für einen NICHT präfixfreien Code:**

```
a -> 0
b -> 01
```
Der codierte String `01` könnte sowohl `ab`... (a gefolgt vom Anfang von b) als auch `b` bedeuten
→ nicht eindeutig decodierbar!

**Beispiel für einen präfixfreien Code:**

```
a -> 0
b -> 10
c -> 11
```
Hier ist keine Codierung der Anfang einer anderen → eindeutig decodierbar, auch ohne Trennzeichen.


In [9]:
def ist_praefixfrei(code: dict) -> bool:
    """Prüft, ob ein Code (Zeichen -> Codewort) präfixfrei ist."""
    woerter = list(code.values())
    for i, w1 in enumerate(woerter):
        for j, w2 in enumerate(woerter):
            if i != j and w2.startswith(w1):
                return False
    return True


code_a = {"a": "0", "b": "01"}
code_b = {"a": "0", "b": "10", "c": "11"}

print(f"Code A {code_a} ist präfixfrei: {ist_praefixfrei(code_a)}")
print(f"Code B {code_b} ist präfixfrei: {ist_praefixfrei(code_b)}")


Code A {'a': '0', 'b': '01'} ist präfixfrei: False
Code B {'a': '0', 'b': '10', 'c': '11'} ist präfixfrei: True


### ✏️ Übung 3
Gegeben sei der Code `{"a": "1", "b": "01", "c": "001", "d": "0001"}`.
1. Ist dieser Code präfixfrei? Überprüfe mit der Funktion `ist_praefixfrei`.
2. Decodiere von Hand die Bitfolge `10010001` mithilfe dieses Codes.


In [10]:
# Übung 3 - Platz zum Ausprobieren
uebung_code = {"a": "1", "b": "01", "c": "001", "d": "0001"}
print(ist_praefixfrei(uebung_code))


True


---
## 4. Das Huffman-Verfahren

Das **Huffman-Verfahren** ist ein verlustfreies Kompressionsverfahren mit **variabler Bitlänge**.
Die Grundidee: Häufig vorkommende Zeichen erhalten **kurze** Codewörter, seltene Zeichen
**lange** Codewörter. Der entstehende Code ist automatisch **präfixfrei**, da jedes Zeichen als
**Blatt** in einem binären **Huffman-Baum** dargestellt wird.

### 4.1 Aufbau des Huffmanbaums (von Hand)

1. Ermittle die Häufigkeit jedes Zeichens im Text.
2. Erzeuge für jedes Zeichen einen Blattknoten mit seiner Häufigkeit.
3. Wiederhole, bis nur noch ein Knoten übrig ist:
   - Wähle die **zwei Knoten mit der geringsten Häufigkeit**.
   - Verbinde sie zu einem neuen Knoten, dessen Häufigkeit die Summe der beiden ist.
4. Der letzte verbleibende Knoten ist die **Wurzel** des Huffmanbaums.
5. Die Codewörter ergeben sich aus dem Pfad von der Wurzel zum jeweiligen Blatt
   (links = 0, rechts = 1).

### Beispiel von Hand: Text `"ABRAKADABRA"`

Häufigkeiten: A=5, B=2, R=2, K=1, D=1

```
Schritt 1: K(1), D(1)  -> zusammenfassen zu KD(2)
Schritt 2: B(2), R(2)  -> zusammenfassen zu BR(4)   (oder KD(2) mit B/R, je nach Tie-Break)
Schritt 3: KD(2), BR? ... (Details hängen von der Tie-Break-Regel ab)
```

Da es bei gleichen Häufigkeiten mehrere gültige Bäume geben kann, ist der Huffman-Code
**nicht eindeutig** – aber die **erwartete Codelänge** ist bei optimaler Wahl immer minimal.

Unten simulieren wir den Algorithmus in Python und zeichnen den Baum als Textstruktur.


In [11]:
import heapq
from collections import Counter

class HuffmanNode:
    def __init__(self, char, freq, left=None, right=None):
        self.char = char      # nur bei Blättern gesetzt
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other):  # für die Heap-Sortierung
        return self.freq < other.freq


def baue_huffman_baum(text: str) -> HuffmanNode:
    haeufigkeiten = Counter(text)
    heap = [HuffmanNode(ch, freq) for ch, freq in haeufigkeiten.items()]
    heapq.heapify(heap)

    if len(heap) == 1:  # Sonderfall: nur ein Zeichen im Text
        einzelknoten = heapq.heappop(heap)
        return HuffmanNode(None, einzelknoten.freq, einzelknoten, None)

    while len(heap) > 1:
        links = heapq.heappop(heap)
        rechts = heapq.heappop(heap)
        neuer_knoten = HuffmanNode(None, links.freq + rechts.freq, links, rechts)
        heapq.heappush(heap, neuer_knoten)

    return heap[0]


def erzeuge_codetabelle(knoten: HuffmanNode, praefix: str = "", tabelle=None) -> dict:
    if tabelle is None:
        tabelle = {}
    if knoten.char is not None:  # Blatt erreicht
        tabelle[knoten.char] = praefix or "0"
        return tabelle
    if knoten.left:
        erzeuge_codetabelle(knoten.left, praefix + "0", tabelle)
    if knoten.right:
        erzeuge_codetabelle(knoten.right, praefix + "1", tabelle)
    return tabelle


def huffman_codieren(text: str):
    baum = baue_huffman_baum(text)
    tabelle = erzeuge_codetabelle(baum)
    codiert = "".join(tabelle[ch] for ch in text)
    return codiert, tabelle, baum


def huffman_decodieren(codiert: str, baum: HuffmanNode) -> str:
    ergebnis = []
    knoten = baum
    for bit in codiert:
        knoten = knoten.left if bit == "0" else knoten.right
        if knoten.char is not None:
            ergebnis.append(knoten.char)
            knoten = baum
    return "".join(ergebnis)


text = "ABRAKADABRA"
codiert, tabelle, baum = huffman_codieren(text)

print(f"Originaltext:      {text}")
print(f"Codetabelle:       {dict(sorted(tabelle.items()))}")
print(f"Codierte Bitfolge: {codiert}")
print(f"Länge codiert:     {len(codiert)} Bit")
print(f"Länge ASCII (8bit):{len(text) * 8} Bit  (zum Vergleich)")

decodiert = huffman_decodieren(codiert, baum)
print(f"\nDecodiert:         {decodiert}")
print(f"Stimmt überein:    {decodiert == text}")


Originaltext:      ABRAKADABRA
Codetabelle:       {'A': '0', 'B': '111', 'D': '100', 'K': '101', 'R': '110'}
Codierte Bitfolge: 01111100101010001111100
Länge codiert:     23 Bit
Länge ASCII (8bit):88 Bit  (zum Vergleich)

Decodiert:         ABRAKADABRA
Stimmt überein:    True


In [12]:
def zeichne_baum(knoten: HuffmanNode, praefix: str = "", ist_links=None):
    """Gibt eine einfache Textdarstellung des Huffmanbaums aus."""
    if knoten is None:
        return
    if knoten.char is not None:
        label = repr(knoten.char)
    else:
        label = f"({knoten.freq})"
    connector = "" if ist_links is None else ("├─0─ " if ist_links else "└─1─ ")
    print(praefix + connector + label + (f"  freq={knoten.freq}" if knoten.char else ""))
    neues_praefix = praefix + ("│    " if ist_links else "     ")
    zeichne_baum(knoten.left, neues_praefix, True)
    zeichne_baum(knoten.right, neues_praefix, False)

print("Huffmanbaum für 'ABRAKADABRA':")
zeichne_baum(baum)


Huffmanbaum für 'ABRAKADABRA':
(11)
     ├─0─ 'A'  freq=5
     └─1─ (6)
          ├─0─ (2)
          │    ├─0─ 'D'  freq=1
          │    └─1─ 'K'  freq=1
          └─1─ (4)
               ├─0─ 'R'  freq=2
               └─1─ 'B'  freq=2


### ✏️ Übung 4
1. Codiere den Text `"MISSISSIPPI"` von Hand: Bestimme die Häufigkeiten, baue den Huffmanbaum
   und lies die Codetabelle ab. Überprüfe dein Ergebnis mit dem Code unten.
2. Vergleiche die Bitlänge der Huffman-Codierung mit einer festen 3-Bit-Codierung
   (da es 5 verschiedene Zeichen gibt, würden 3 Bit pro Zeichen reichen). Wie viel Prozent
   spart Huffman ein?
3. Warum ist Huffman-Codierung bei einem Text mit **gleichverteilten** Zeichenhäufigkeiten
   kaum von Vorteil gegenüber fester Bitlänge?


In [13]:
# Übung 4 - Platz zum Ausprobieren
text2 = "MISSISSIPPI"
codiert2, tabelle2, baum2 = huffman_codieren(text2)
print(f"Codetabelle: {dict(sorted(tabelle2.items()))}")
print(f"Codierte Länge: {len(codiert2)} Bit")
print(f"Feste 3-Bit-Codierung wäre: {len(text2) * 3} Bit")
zeichne_baum(baum2)


Codetabelle: {'I': '0', 'M': '100', 'P': '101', 'S': '11'}
Codierte Länge: 21 Bit
Feste 3-Bit-Codierung wäre: 33 Bit
(11)
     ├─0─ 'I'  freq=4
     └─1─ (7)
          ├─0─ (3)
          │    ├─0─ 'M'  freq=1
          │    └─1─ 'P'  freq=2
          └─1─ 'S'  freq=4


---
## 5. Festkommadarstellung (Leistungsfach)

Um **nichtganze Zahlen** (Zahlen mit Nachkommastellen) binär darzustellen, ist eine einfache
Möglichkeit die **Festkommadarstellung** (fixed point): Man legt eine feste Anzahl von Bits für
den Vorkomma- und eine feste Anzahl für den Nachkommateil fest.

### Beispiel: Q4.4-Format (4 Bit Vorkomma, 4 Bit Nachkomma)

Die Nachkommabits repräsentieren Zweierpotenzen mit **negativen** Exponenten:

$$ b_3 b_2 b_1 b_0 \,.\, b_{-1} b_{-2} b_{-3} b_{-4} $$

$$ \text{Wert} = b_3\cdot2^3 + b_2\cdot2^2 + b_1\cdot2^1 + b_0\cdot2^0
+ b_{-1}\cdot2^{-1} + b_{-2}\cdot2^{-2} + b_{-3}\cdot2^{-3} + b_{-4}\cdot2^{-4} $$

**Beispiel:** `0101.1100` = $4 + 1 + 0{,}5 + 0{,}25 = 5{,}75$

### Vor- und Nachteile
- ✅ Einfache, schnelle Arithmetik (wie bei Ganzzahlen)
- ❌ Fester, meist kleiner Wertebereich
- ❌ Feste Genauigkeit – nicht jede Dezimalzahl ist exakt darstellbar (z. B. 0,1 in Binär)

*(Im Gegensatz dazu steht die **Gleitkommadarstellung**, z. B. IEEE 754, die Vor- und Nachkommateil
dynamisch über einen Exponenten verschiebt und damit einen viel größeren Wertebereich abdeckt.)*


In [14]:
def festkomma_dezimal(bitstring: str, vorkomma_bits: int, nachkomma_bits: int) -> float:
    """Wandelt einen Festkomma-Bitstring in eine Dezimalzahl um (ohne Vorzeichen)."""
    assert len(bitstring) == vorkomma_bits + nachkomma_bits
    vor_teil = bitstring[:vorkomma_bits]
    nach_teil = bitstring[vorkomma_bits:]

    ganzzahl_wert = int(vor_teil, 2)
    nachkomma_wert = sum(int(bit) * 2 ** (-(i + 1)) for i, bit in enumerate(nach_teil))
    return ganzzahl_wert + nachkomma_wert


beispiel = "01011100"  # Q4.4
wert = festkomma_dezimal(beispiel, 4, 4)
print(f"{beispiel} (Q4.4) = {wert}")

for bs in ["00000001", "00001000", "11111111"]:
    print(f"{bs} (Q4.4) = {festkomma_dezimal(bs, 4, 4)}")


01011100 (Q4.4) = 5.75
00000001 (Q4.4) = 0.0625
00001000 (Q4.4) = 0.5
11111111 (Q4.4) = 15.9375


### ✏️ Übung 5
1. Wandle die Dezimalzahl **9,25** in das Q4.4-Festkommaformat um (von Hand).
2. Welche kleinste positive Zahl größer als 0 lässt sich im Q4.4-Format darstellen?
   Welche größte Zahl?
3. Kann die Dezimalzahl **0,1** exakt im Q4.4-Format dargestellt werden? Probiere es mit der
   Funktion `festkomma_dezimal` für verschiedene Nachkommabits aus.


---
## 6. Erweiterungsmöglichkeiten von Textcodierungen (Leistungsfach)

Der klassische **ASCII-Code** verwendet 7 Bit und deckt damit 128 Zeichen ab (lateinisches
Alphabet, Ziffern, Satzzeichen, Steuerzeichen). Für weitere Zeichen (Umlaute, Sonderzeichen,
andere Schriftsysteme) braucht man Erweiterungsmöglichkeiten:

### 6.1 Biterweiterung
Man fügt dem Code **weitere Bits** hinzu, um mehr Zeichen darstellen zu können.
Beispiel: ASCII (7 Bit, 128 Zeichen) → **Latin-1 / ISO 8859-1** (8 Bit, 256 Zeichen) →
**Unicode** (bis zu 32 Bit / UTF-32, über 1 Million Zeichen möglich).

### 6.2 Escapezeichen
Ein spezielles Steuerzeichen (Escape-Sequenz) signalisiert, dass die nachfolgenden Bits/Bytes
**anders interpretiert** werden sollen als normale Zeichen. So lassen sich mit derselben
Bitlänge zusätzliche Bedeutungen codieren, ohne die Wortlänge zu erhöhen. Beispiel: In UTF-8
signalisieren bestimmte Bitmuster im ersten Byte, dass ein Zeichen aus **mehreren Bytes**
besteht (Mehrbyte-Escape).

### 6.3 Codepages
Eine **Codepage** ist eine feste Zuordnungstabelle zwischen Bitmustern (meist 8 Bit) und
Zeichen für eine bestimmte Sprache/Region (z. B. Codepage 1252 für Westeuropa,
Codepage 1251 für Kyrillisch). Das Problem: Dasselbe Bitmuster bedeutet je nach verwendeter
Codepage ein **anderes Zeichen** – das führt zu Darstellungsfehlern ("Zeichensalat"), wenn
Sender und Empfänger unterschiedliche Codepages verwenden. **Unicode/UTF-8** löst dieses
Problem, indem es weltweit einheitlich verwendet werden kann.


In [15]:
# Demonstration: Biterweiterung -- gleiche Bitfolge, unterschiedliche Interpretation je nach Codepage
byte_wert = 0xE9  # 233 dezimal

print(f"Bitmuster: {byte_wert:08b} (0x{byte_wert:02X})")
print(f"  Interpretiert als Latin-1 (ISO-8859-1): {bytes([byte_wert]).decode('latin-1')!r}")
print(f"  Interpretiert als CP437 (DOS):          {bytes([byte_wert]).decode('cp437')!r}")

# UTF-8: Zeichen ausserhalb des ASCII-Bereichs benoetigen mehrere Bytes ("Escape"-Praefixe)
zeichen = "€"  # Euro-Zeichen, nicht in ASCII enthalten
utf8_bytes = zeichen.encode("utf-8")
print(f"\n'{zeichen}' in UTF-8: {[format(b, '08b') for b in utf8_bytes]}")
print("Das erste Byte beginnt mit '1110', ein Escape-Praefix, das signalisiert:")
print("'Dieses Zeichen besteht aus insgesamt 3 Bytes.'")


Bitmuster: 11101001 (0xE9)
  Interpretiert als Latin-1 (ISO-8859-1): 'é'
  Interpretiert als CP437 (DOS):          'Θ'

'€' in UTF-8: ['11100010', '10000010', '10101100']
Das erste Byte beginnt mit '1110', ein Escape-Praefix, das signalisiert:
'Dieses Zeichen besteht aus insgesamt 3 Bytes.'


### ✏️ Übung 6
1. Warum kann es beim Öffnen einer alten Textdatei zu falsch dargestellten Umlauten kommen,
   wenn die Codepage nicht bekannt ist?
2. Erkläre in eigenen Worten, wie UTF-8 mit Escapezeichen/-präfixen eine variable Anzahl von
   Bytes pro Zeichen ermöglicht, obwohl reines ASCII nur 1 Byte benötigt.


---
## 7. Das LZW-Verfahren (Leistungsfach)

**LZW (Lempel-Ziv-Welch)** ist ein **Wörterbuchverfahren** zur verlustfreien Kompression.
Im Gegensatz zu Huffman (das auf Zeichenhäufigkeiten basiert) baut LZW während der Codierung
**dynamisch ein Wörterbuch** aus bereits gesehenen Zeichenfolgen auf und ersetzt wiederkehrende
Folgen durch kurze Referenzen (Wörterbuch-Indizes).

### 7.1 Codierung – Ablauf (von Hand)

1. Initialisiere das Wörterbuch mit allen Einzelzeichen (z. B. ASCII: 0–255).
2. Lies das längste bekannte Präfix `W` aus dem Text, für das `W` bereits im Wörterbuch steht.
3. Gib den Index von `W` aus.
4. Füge `W + nächstes Zeichen` als **neuen Eintrag** ins Wörterbuch ein.
5. Fahre mit dem nächsten Zeichen fort, bis der Text verarbeitet ist.

### Beispiel von Hand: Text `"ABABABA"`  (Wörterbuch startet bei A=1, B=2)

| Schritt | gelesen | Ausgabe | neuer Wörterbucheintrag |
|---|---|---|---|
| 1 | A | 1 (A) | AB -> 3 |
| 2 | B | 2 (B) | BA -> 4 |
| 3 | AB | 3 (AB) | ABA -> 5 |
| 4 | AB (Rest: A) | ... | ... |

*(Die genaue Fortführung wird im Python-Code unten simuliert und ausgegeben.)*


In [16]:
def lzw_codieren(text: str):
    # Woerterbuch mit allen Einzelzeichen initialisieren
    woerterbuch = {chr(i): i for i in range(256)}
    naechster_code = 256

    w = ""
    ausgabe = []
    protokoll = []  # fuer die Nachvollziehbarkeit

    for zeichen in text:
        wc = w + zeichen
        if wc in woerterbuch:
            w = wc
        else:
            ausgabe.append(woerterbuch[w])
            protokoll.append((w, woerterbuch[w], wc, naechster_code))
            woerterbuch[wc] = naechster_code
            naechster_code += 1
            w = zeichen

    if w:
        ausgabe.append(woerterbuch[w])
        protokoll.append((w, woerterbuch[w], None, None))

    return ausgabe, protokoll


text = "ABABABA"
ausgabe, protokoll = lzw_codieren(text)

print(f"Text: {text!r}\n")
print(f"{'gelesen':<10}{'Code (aus)':<12}{'neuer Eintrag':<15}{'neuer Code'}")
for w, code_, neu, neuer_code in protokoll:
    neu_str = f"{neu!r}" if neu else "-"
    neuer_code_str = str(neuer_code) if neuer_code else "-"
    print(f"{w!r:<10}{code_:<12}{neu_str:<15}{neuer_code_str}")

print(f"\nCodierte Ausgabe (Indizes): {ausgabe}")
print(f"Original: {len(text)} Zeichen -> Codiert: {len(ausgabe)} Codes")


Text: 'ABABABA'

gelesen   Code (aus)  neuer Eintrag  neuer Code
'A'       65          'AB'           256
'B'       66          'BA'           257
'AB'      256         'ABA'          258
'ABA'     258         -              -

Codierte Ausgabe (Indizes): [65, 66, 256, 258]
Original: 7 Zeichen -> Codiert: 4 Codes


In [17]:
def lzw_decodieren(codes: list) -> str:
    woerterbuch = {i: chr(i) for i in range(256)}
    naechster_code = 256

    w = woerterbuch[codes[0]]
    ergebnis = [w]

    for code_ in codes[1:]:
        if code_ in woerterbuch:
            eintrag = woerterbuch[code_]
        elif code_ == naechster_code:
            eintrag = w + w[0]   # Sonderfall
        else:
            raise ValueError("Ungueltiger Code")

        ergebnis.append(eintrag)
        woerterbuch[naechster_code] = w + eintrag[0]
        naechster_code += 1
        w = eintrag

    return "".join(ergebnis)


decodiert = lzw_decodieren(ausgabe)
print(f"Decodiert: {decodiert!r}")
print(f"Stimmt mit Original überein: {decodiert == text}")


Decodiert: 'ABABABA'
Stimmt mit Original überein: True


### ✏️ Übung 7
1. Führe die LZW-Codierung von Hand für den Text `"AAAAAA"` durch und überprüfe dein Ergebnis
   mit `lzw_codieren("AAAAAA")`.
2. Vergleiche LZW und Huffman: Bei welcher Art von Text (z. B. viele Wiederholungen vs.
   gleichverteilte Zeichen) ist welches Verfahren im Vorteil?


In [18]:
# Übung 7 - Platz zum Ausprobieren
ausgabe2, protokoll2 = lzw_codieren("AAAAAA")
print(ausgabe2)


[65, 256, 257]


---
## 8. Hashfunktionen (Leistungsfach)

Eine **Hashfunktion** bildet Eingabedaten beliebiger Länge auf einen **Hashwert fester Länge**
(den "Fingerabdruck") ab.

### 8.1 Anforderungen an (kryptographische) Hashfunktionen
- **Determinismus:** Dieselbe Eingabe liefert immer denselben Hashwert.
- **Nichtumkehrbarkeit (Einwegfunktion):** Aus dem Hashwert darf die ursprüngliche Eingabe
  praktisch nicht rekonstruierbar sein.
- **Kollisionsresistenz:** Es soll praktisch unmöglich sein, zwei unterschiedliche Eingaben
  zu finden, die denselben Hashwert erzeugen (Kollisionen).
- **Lawineneffekt:** Eine minimale Änderung der Eingabe (z. B. 1 Bit) führt zu einem
  komplett anderen Hashwert.

### 8.2 Anwendungsbeispiele
- **Fingerprint / Integritätsprüfung:** Prüfsummen von Dateien vergleichen, um Manipulationen
  oder Übertragungsfehler zu erkennen.
- **ISBN-Prüfsumme:** Eine einfache Prüfziffer erkennt Tippfehler bei der Eingabe einer ISBN.
- **Passworthashes:** Passwörter werden nicht im Klartext, sondern als Hashwert gespeichert.
- **Digitale Signaturen:** Es wird nicht die ganze Nachricht, sondern nur ihr Hashwert
  signiert (Effizienz).

⚠️ Hinweis: Python-`hash()` und einfache Prüfsummen wie CRC sind **nicht** kollisionsresistent
und daher **nicht** für Sicherheitszwecke geeignet. Für kryptographische Zwecke nutzt man
Verfahren wie SHA-256.


In [19]:
import hashlib

nachricht = "Hallo Welt"
nachricht_veraendert = "Hallo welt"  # nur ein Buchstabe geaendert!

hash1 = hashlib.sha256(nachricht.encode()).hexdigest()
hash2 = hashlib.sha256(nachricht_veraendert.encode()).hexdigest()

print(f"Nachricht 1: {nachricht!r}")
print(f"SHA-256:     {hash1}\n")

print(f"Nachricht 2: {nachricht_veraendert!r}")
print(f"SHA-256:     {hash2}\n")

print("--> Lawineneffekt: Eine winzige Änderung der Eingabe führt zu einem komplett anderen Hash!")


Nachricht 1: 'Hallo Welt'
SHA-256:     2d2da19605a34e037dbe82173f98a992a530a5fdd53dad882f570d4ba204ef30

Nachricht 2: 'Hallo welt'
SHA-256:     a1401e39ea9735fdcebc52013babcc1143ff56664e025cae31b4323382e16e57

--> Lawineneffekt: Eine winzige Änderung der Eingabe führt zu einem komplett anderen Hash!


In [20]:
def isbn10_pruefziffer_gueltig(isbn: str) -> bool:
    """Prueft eine 10-stellige ISBN mithilfe der gewichteten Pruefsumme."""
    ziffern = isbn.replace("-", "")
    if len(ziffern) != 10:
        return False
    summe = 0
    for position, zeichen in enumerate(ziffern):
        wert = 10 if zeichen.upper() == "X" and position == 9 else int(zeichen)
        gewicht = 10 - position
        summe += wert * gewicht
    return summe % 11 == 0


# Beispiel: gueltige ISBN
beispiele = ["0-306-40615-2", "0-306-40615-3"]  # zweite ist absichtlich falsch (letzte Ziffer geaendert)
for isbn in beispiele:
    print(f"{isbn}: gültig = {isbn10_pruefziffer_gueltig(isbn)}")


0-306-40615-2: gültig = True
0-306-40615-3: gültig = False


### ✏️ Übung 8
1. Ändere im obigen Code die Nachricht geringfügig (z. B. ein Leerzeichen hinzufügen) und
   beobachte, wie stark sich der SHA-256-Hash verändert.
2. Warum eignet sich eine Hashfunktion **nicht**, um ein verschlüsseltes Passwort später wieder
   zu entschlüsseln?
3. Finde (händisch oder mit Code) heraus, ob es zu deinem Namen eine andere Zeichenkette gibt,
   die denselben `hash()`-Wert in Python erzeugt (Kollision). Was sagt dir das Ergebnis über
   die Kollisionsresistenz von Python's eingebauter `hash()`-Funktion?


In [21]:
# Übung 8 - Platz zum Ausprobieren
print(hashlib.sha256("Hallo Welt ".encode()).hexdigest())  # mit Leerzeichen am Ende


bd03d06734202effc507c32446d7c13208dae7cc45d52b2748c3ca5985d0fefb


---
## Zusammenfassung & Selbstkontrolle

| Kompetenz | Abschnitt |
|---|---|
| (1) Einer-/Zweierkomplement | 1 |
| (2) Addition/Subtraktion im Zweierkomplement | 2 |
| (3) Overflow, endliche Stellenzahl | 2 |
| (4) Festkommadarstellung *(L)* | 5 |
| (5) Erweiterungen von Textcodierungen *(L)* | 6 |
| (6) Merkmale von Codes | 3 |
| (7) Huffman-Verfahren | 4 |
| (8) LZW-Verfahren *(L)* | 7 |
| (9) Hashfunktionen *(L)* | 8 |

*(L) = nur Leistungsfach*

**Checkliste – kannst du...**
- [ ] eine Zahl im Zweierkomplement in beide Richtungen umrechnen?
- [ ] von Hand zwei Zweierkomplement-Zahlen addieren und einen Overflow erkennen?
- [ ] erklären, was Präfixfreiheit bedeutet und warum sie für Huffman wichtig ist?
- [ ] einen Huffmanbaum von Hand aufbauen und damit codieren/decodieren?
- [ ] eine Dezimalzahl in Festkommadarstellung umwandeln?
- [ ] den Unterschied zwischen Biterweiterung, Escapezeichen und Codepages erklären?
- [ ] das LZW-Verfahren von Hand nachvollziehen?
- [ ] die Anforderungen an eine Hashfunktion nennen und Anwendungsbeispiele geben?

Viel Erfolg beim Üben! 🎓
